# Clustering Hiérarchique

**ifri-mini-ml-lib · Module `clustering` · `HierarchicalClustering`**

---

### Plan

1. [Théorie](#1.-Théorie)
2. [L'algorithme pas à pas](#2.-L'algorithme-pas-à-pas)
3. [Les critères de linkage](#3.-Les-critères-de-linkage)
4. [Démo interactive — Iris](#4.-Démo-interactive-—-Iris)
5. [Le Dendrogramme](#5.-Le-Dendrogramme)
6. [Agglomératif vs Divisif](#6.-Agglomératif-vs-Divisif)
7. [Avantages et limites](#7.-Avantages-et-limites)

## 1. Théorie

### Qu'est-ce que le clustering hiérarchique ?

Le **clustering hiérarchique** est une méthode d'apprentissage non supervisé qui construit une
arborescence de clusters appelée **dendrogramme**, sans qu'on ait besoin de fixer le nombre
de clusters à l'avance.

Contrairement à K-Means qui produit une partition plate en $k$ groupes, le clustering
hiérarchique organise les données en **niveaux imbriqués** : un cluster peut contenir
des sous-clusters, qui eux-mêmes contiennent des sous-sous-clusters, etc.

---

### Deux stratégies opposées

| | **Agglomératif** *(bottom-up)* | **Divisif** *(top-down)* |
|---|---|---|
| **Départ** | Chaque point = 1 cluster | Tous les points = 1 cluster |
| **Opération** | Fusionne les 2 clusters les plus proches | Divise le plus grand cluster |
| **Fin** | Un seul cluster (ou k clusters) | k clusters atteint |
| **Complexité** | $O(n^3)$ naïf, $O(n^2 \log n)$ optimisé | $O(n^2 k)$ via bisection |
| **Usage courant** | Très répandu | Plus rare |

Dans `ifri_mini_ml_lib`, l'approche **agglomérative** utilise les distances exactes,
et l'approche **divisive** utilise K-Means (k=2) pour bissecter les clusters.

## 2. L'algorithme pas à pas

### Algorithme agglomératif

Soit $\mathcal{X} = \{x_1, x_2, \ldots, x_n\}$ un ensemble de $n$ points.

**Initialisation :** $n$ clusters singletons $C_1 = \{x_1\}, C_2 = \{x_2\}, \ldots, C_n = \{x_n\}$

**Itération :** À chaque étape $t$ :

$$
(C_i^*, C_j^*) = \arg\min_{i \neq j} \ d_{\text{link}}(C_i, C_j)
$$

Fusionner : $C_{\text{new}} = C_i^* \cup C_j^*$, retirer $C_i^*$ et $C_j^*$.

**Arrêt :** Quand il reste $k$ clusters (ou 1 seul pour le dendrogramme complet).

---

### Exemple visuel sur 6 points

```
Étape 0 : {A} {B} {C} {D} {E} {F}   (6 clusters)
Étape 1 : {A,B} {C} {D} {E} {F}     (merge A et B : les plus proches)
Étape 2 : {A,B} {C} {D,E} {F}       (merge D et E)
Étape 3 : {A,B,C} {D,E} {F}         (merge {A,B} et C)
Étape 4 : {A,B,C} {D,E,F}           (merge {D,E} et F)  → 2 clusters
```

## 3. Les critères de linkage

La fonction $d_{\text{link}}(C_i, C_j)$ définit comment mesurer la "distance" entre
deux clusters. C'est le choix le plus important de l'algorithme.

Soient $C_i$ et $C_j$ deux clusters. La distance Euclidienne entre deux points est :
$d(x, y) = \|x - y\|_2$

---

### Single linkage (minimum)

$$d_{\text{single}}(C_i, C_j) = \min_{x \in C_i,\ y \in C_j} d(x, y)$$

La distance entre deux clusters = la distance entre leurs **points les plus proches**.

→ Produit des clusters en « chaîne » (chaining effect). Sensible aux outliers.

---

### Complete linkage (maximum)

$$d_{\text{complete}}(C_i, C_j) = \max_{x \in C_i,\ y \in C_j} d(x, y)$$

La distance entre deux clusters = la distance entre leurs **points les plus éloignés**.

→ Produit des clusters compacts et de taille similaire. Plus robuste aux outliers.

---

### Average linkage (UPGMA)

$$d_{\text{average}}(C_i, C_j) = \frac{1}{|C_i| \cdot |C_j|} \sum_{x \in C_i} \sum_{y \in C_j} d(x, y)$$

La distance = **moyenne de toutes les distances inter-clusters**.

→ Compromis entre single et complete. Généralement le plus robuste.

## 4. Démo interactive — Iris

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from ifri_mini_ml_lib.clustering import HierarchicalClustering
from ifri_mini_ml_lib.metrics.clustering import calculate_silhouette
from notebooks.clustering.utils import load_hierarchical_iris_data

In [ ]:
# Chargement des données Iris via le helper partagé
iris = load_hierarchical_iris_data()
X = iris['data']
y_true = iris['target']

# On garde les 2 premières features pour la visualisation 2D
X_2d = X[:, :2]

print(f"Dataset Iris chargé : {X.shape[0]} points, {X.shape[1]} features")
print(f"Classes réelles : {iris['target_names']}")

## 5. Le Dendrogramme

Le **dendrogramme** est la représentation graphique de la hiérarchie de clusters.
C'est l'outil clé pour **choisir le nombre de clusters** sans le fixer à l'avance.

### Comment lire un dendrogramme ?

- **Axe horizontal** : les points (feuilles) et les clusters fusionnés
- **Axe vertical** : la distance à laquelle s'effectue chaque fusion
- **Couper horizontalement** à une certaine hauteur $h$ donne un nombre de clusters :
  le nombre de branches coupées = le nombre de clusters

> **Règle pratique** : chercher le **plus grand saut vertical** dans le dendrogramme.
> Ce saut indique qu'on a fusionné deux clusters vraiment distincts — c'est là qu'on devrait couper.

In [ ]:
# Dendrogramme interactif
w_dend_linkage = widgets.ToggleButtons(
    options=['single', 'complete', 'average'],
    value='complete',
    description='Linkage :',
    button_style='warning',
    style={'description_width': 'initial'},
)

w_dend_k = widgets.IntSlider(
    value=3, min=2, max=6, step=1,
    description='k (clusters colorés) :',
    style={'description_width': 'initial'},
)

w_dend_n = widgets.IntSlider(
    value=30, min=10, max=150, step=10,
    description='Nb points utilisés :',
    style={'description_width': 'initial'},
)

out_dend = widgets.Output()

def run_dend(change=None):
    with out_dend:
        clear_output(wait=True)
        n = w_dend_n.value
        linkage = w_dend_linkage.value
        k = w_dend_k.value

        # Sous-échantillon stratifié pour garder les 3 classes
        np.random.seed(42)
        idx = []
        for c in range(3):
            class_idx = np.where(y_true == c)[0]
            idx.extend(np.random.choice(class_idx, size=n // 3, replace=False))
        idx = np.array(idx)
        X_sub = X[:, [2, 3]][idx]
        y_sub = y_true[idx]

        hc = HierarchicalClustering(n_clusters=k, linkage=linkage)
        hc.fit_predict(X_sub)

        # Utilise le plot_dendrogram de la lib
        species_labels = [iris['target_names'][y_sub[i]] for i in range(len(idx))]
        plt.figure(figsize=(14, 5))
        hc.plot_dendrogram(labels=species_labels)

for w in [w_dend_linkage, w_dend_k, w_dend_n]:
    w.observe(run_dend, names='value')

display(widgets.VBox([
    widgets.HTML('<h4>🌳 Dendrogramme interactif</h4>'),
    w_dend_linkage, w_dend_k, w_dend_n,
    out_dend
]))
run_dend()

## 6. Agglomératif vs Divisif

In [ ]:
import time

from notebooks.clustering.utils import plot_hierarchical_method_comparison

# Comparaison côte à côte sur le même sous-échantillon
np.random.seed(0)
idx50 = np.random.choice(len(X), 50, replace=False)
X50 = X[:, [2, 3]][idx50]
y50 = y_true[idx50]

results = []
for method, title in zip(
    ['agglomerative', 'divisive'],
    ['Agglomératif (bottom-up)', 'Divisif (top-down)']
):
    t0 = time.time()
    hc = HierarchicalClustering(n_clusters=3, linkage='complete', method=method)
    labels = hc.fit_predict(X50)
    elapsed = time.time() - t0
    sil = calculate_silhouette(X50, labels)

    results.append({
        'X': X50,
        'labels': labels,
        'title': title,
        'silhouette': sil,
        'elapsed': elapsed,
        'xlabel': 'Petal length (cm)',
        'ylabel': 'Petal width (cm)',
    })

plot_hierarchical_method_comparison(results)

## 7. Avantages et limites

| | Avantages | Limites |
|---|---|---|
| **Pas de k à fixer** | On peut explorer le dendrogramme pour choisir k | Le dendrogramme peut être difficile à interpréter sur de grands datasets |
| **Déterministe** | Pas d'aléatoire (agglomératif) → résultat reproductible | $O(n^3)$ en temps et $O(n^2)$ en mémoire : lent sur >1000 points |
| **Dendrogramme** | Vue hiérarchique complète de la structure des données | Sensibilité au linkage : le choix du critère change fortement les résultats |
| **Formes arbitraires** | Single linkage peut détecter des formes non convexes | Pas de notion de « centroïde » → difficile à interpréter |

---

### Quand utiliser le clustering hiérarchique ?

- Quand on **ne sait pas** combien de clusters chercher (utiliser le dendrogramme)
- Quand les données sont de **petite taille** (< quelques milliers de points)
- Quand on veut une **vue hiérarchique** de la structure (taxonomie, phylogénie, etc.)
- Pour la **visualisation** et l'exploration de données

### Quand préférer K-Means ?

- Grands datasets (>10 000 points)
- Clusters sphériques et de taille similaire
- Besoin de rapidité d'exécution

## 8. Utilisation pratique

### Exemple simple : Démarrage rapide

Voici comment utiliser le clustering hiérarchique en quelques lignes :

In [ ]:
from ifri_mini_ml_lib.clustering import HierarchicalClustering
from sklearn.datasets import make_blobs
import numpy as np

# 1. Générer des données de synthèse
X, y_true = make_blobs(n_samples=100, centers=4, n_features=2, 
                        random_state=42, cluster_std=0.8)

# 2. Appliquer le clustering hiérarchique agglomératif
hc = HierarchicalClustering(n_clusters=4, linkage='complete', method='agglomerative')
labels = hc.fit_predict(X)

# 3. Afficher les résultats
print(f"Nombre de clusters trouvés : {len(set(labels))}")
print(f"Étiquettes : {labels}")

# 4. Visualiser
hc.plot_clusters(X)

### Choisir le bon paramètre : `linkage`

Chaque critère de linkage donne des résultats différents. Voici comment choisir :

In [ ]:
from notebooks.clustering.utils import plot_hierarchical_linkage_comparison

# Comparer les trois critères sur les mêmes données
linkages = ['single', 'complete', 'average']
results = []

for link in linkages:
    hc = HierarchicalClustering(n_clusters=4, linkage=link, method='agglomerative')
    labels = hc.fit_predict(X)
    sil = calculate_silhouette(X, labels)

    results.append({
        'linkage': link,
        'labels': labels,
        'silhouette': sil,
    })

plot_hierarchical_linkage_comparison(X, results)

print("💡 Résumé :")
print("  • 'single'   : Bonne détection des formes non-convexes, sensible aux outliers")
print("  • 'complete' : Clusters compacts et balancés, robuste aux outliers")
print("  • 'average'  : Compromis équilibré (généralement recommandé)")

### Quand utiliser agglomératif vs divisif ?

| Situation | Recommandation | Raison |
|-----------|---|---|
| Vous avez < 1000 points | **Agglomératif** | Plus rapide, dendrogramme disponible |
| Vous avez > 10 000 points | **Divisif** | Moins de mémoire, plus rapide |
| Vous ne savez pas combien de clusters | **Agglomératif** | Le dendrogramme guide le choix |
| Les clusters sont sphériques | **Indifférent** | Les deux marchent bien |
| Clusters de formes variées | **Single linkage** | Peut détecter des formes complexes |

## 9. Applications réelles

### 1. **Biologie / Génétique**
Les dendrogrammes hiérarchiques sont utilisés pour construire des **arbres phylogénétiques** : 
classifier les espèces selon leurs similarités génétiques. Chaque feuille = une espèce, 
et la hauteur de fusion indique le temps d'divergence évolutif.

### 2. **Marketing et segmentation client**
Les entreprises utilisent le clustering hiérarchique pour segmenter leur base clients 
selon le comportement d'achat, la démographie, etc. Le dendrogramme aide à explorer 
différents niveaux de segmentation (ex: clients ultra-fidèles → fidèles → occasionnels).

### 3. **Traitement du langage naturel (NLP)**
Dans l'analyse de documents ou de textes, on peut :
- Grouper des documents similaires hiérarchiquement
- Créer des hiérarchies de thèmes (ex: sport → football → équipes de ligue 1)
- Analyser l'évolution des sujets au fil du temps

### 4. **Imagerie médicale**
Les médecins utilisent le clustering hiérarchique pour grouper des scans (IRM, CT, radiographies) 
en fonction de la sévérité des symptômes ou du type de maladie détecté.

### 5. **Écologie et biologie comportementale**
Classification hiérarchique des espèces animales selon leur comportement, habitat, régime alimentaire.
Permet de découvrir des regroupements naturels sans les imposer à l'avance.

## 10. Points clés à retenir

✅ **Avantages du clustering hiérarchique :**
- ✓ Pas besoin de fixer $k$ à l'avance
- ✓ Dendrogramme donne une vue hiérarchique complète
- ✓ Déterministe (agglomératif) → résultats reproductibles
- ✓ Peut détecter des structures non-convexes (single linkage)

⚠️ **Limitations importantes :**
- ✗ Complexité $O(n^3)$ → lent sur gros datasets (> 10 000 points)
- ✗ Impossible de revenir en arrière : une fusion est définitive
- ✗ Très sensible au choix du linkage
- ✗ Dendrogramme difficile à interpréter visuellement au-delà de ~100 points

🎯 **Meilleurs cas d'usage :**
- Petits à moyens datasets (< 1000 points)
- Exploration de données (on veut explorer plusieurs k)
- Structures hiérarchiques naturelles (phylogénie, taxonomie, etc.)
- Visualisation et compréhension des données

## 11. Références

### Articles et ressources
- **Hierarchical Clustering** — Introduction to Statistical Learning (ISLR)
  https://www.statlearning.com/
  
- **Agglomerative Hierarchical Clustering Algorithms** — Müllner, 2011
  https://arxiv.org/pdf/1109.2378.pdf
  
- **An Introduction to Hierarchical Clustering** — StatQuest with Josh Starmer (YouTube)
  https://youtu.be/7xHsRkOdVKc

### Bibliothèques apparentées
- **SciPy** : `scipy.cluster.hierarchy` — implémentation de référence avec dendrogrammes avancés
- **scikit-learn** : `AgglomerativeClustering` — version scikit-learn
- **seaborn** : `clustermap()` — dendrogramme avec heatmap intégré

### Jeux de données pour pratiquer
- **Iris** : Dataset classique, petite taille, parfait pour apprendre
- **Digits** : Images manuscrites, 1797 points, 64 features
- **Wine** : Classification de vins, 178 samples, 13 features
- **Custom** : Générer avec `sklearn.datasets.make_blobs()` ou `make_moons()`